In [3]:
!pip install pyspark psycopg2 sqlalchemy pandas

In [4]:
from pyspark.sql import SparkSession
import psycopg2
import pandas as pd
from pyspark.sql.functions import col, when, trim, lower, upper, regexp_replace, to_date, concat_ws, monotonically_increasing_id, year, month, day, sum, count, round, lit
from sqlalchemy import create_engine

In [5]:
spark =  SparkSession.builder\
         .appName("DealWithus")\
         .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0")\
         .getOrCreate()

In [6]:
df = spark.read.csv(r"C:\Users\Admin\Desktop\10Alytics\DealWithUs\DealWithUs\Raw_Data\dealwithus_raw_data.csv", header = True, inferSchema = True)

In [7]:
df.printSchema()

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- CustomerEmail: string (nullable = true)
 |-- CustomerPhone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- Last Name: string (nullable = true)



In [8]:
df.count()

299966

In [9]:
df.columns

['OrderID',
 'ProductID',
 'Quantity',
 'CustomerID',
 'OrderDate',
 'PaymentMethod',
 'UnitPrice',
 'OrderStatus',
 'TotalAmount',
 'CustomerEmail',
 'CustomerPhone',
 'City',
 'Country',
 'ProductName',
 'Category',
 'FirstName',
 'Last Name']

In [10]:
for column in df.columns:
    print(f"{column} Nulls: {df.filter(df[column].isNull()).count()}")

OrderID Nulls: 0
ProductID Nulls: 0
Quantity Nulls: 0
CustomerID Nulls: 0
OrderDate Nulls: 0
PaymentMethod Nulls: 172
UnitPrice Nulls: 0
OrderStatus Nulls: 0
TotalAmount Nulls: 0
CustomerEmail Nulls: 0
CustomerPhone Nulls: 0
City Nulls: 130
Country Nulls: 0
ProductName Nulls: 0
Category Nulls: 0
FirstName Nulls: 0
Last Name Nulls: 0


In [11]:
df = df.fillna({"PaymentMethod":"Unknown",
                "City":"Unknown"})

In [12]:
df = df.withColumn('Year', year(to_date(col('OrderDate'), 'MM/dd/yyyy')))
df = df.withColumn('Month', month(to_date(col('OrderDate'), 'MM/dd/yyyy')))
df = df.withColumn('Day', day(to_date(col('OrderDate'), 'MM/dd/yyyy')))

In [13]:
df.show(5)


+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+----+-----+---+
|OrderID|ProductID|Quantity|CustomerID| OrderDate|PaymentMethod|UnitPrice|OrderStatus|TotalAmount|       CustomerEmail|     CustomerPhone|             City|             Country|       ProductName| Category|FirstName|Last Name|Year|Month|Day|
+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+----+-----+---+
|  O0094|    P0001|       1|     C1954|2024-12-02|       PayPal|   1491.0|    Pending|    8476.45|griffinmichelle@e...|     (808)782-4405|       Emilymouth|Northern Mariana ...|Interesting Tablet|Computers|   Kelsey|   Burton|2024|   12|  2|
|  O0312|    P0001|       2|    

In [14]:
df = df.withColumn('Revenue', (col('Quantity')*col('UnitPrice')))

In [15]:
df.show(5)

+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+----+-----+---+-------+
|OrderID|ProductID|Quantity|CustomerID| OrderDate|PaymentMethod|UnitPrice|OrderStatus|TotalAmount|       CustomerEmail|     CustomerPhone|             City|             Country|       ProductName| Category|FirstName|Last Name|Year|Month|Day|Revenue|
+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+----+-----+---+-------+
|  O0094|    P0001|       1|     C1954|2024-12-02|       PayPal|   1491.0|    Pending|    8476.45|griffinmichelle@e...|     (808)782-4405|       Emilymouth|Northern Mariana ...|Interesting Tablet|Computers|   Kelsey|   Burton|2024|   12|  2| 1491.0|


In [16]:
df_new = df.filter(col('OrderStatus') == 'Cancelled')

In [17]:
df_new.show(5)

+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+------------------+----------------+------------------+---------+---------+---------+----+-----+---+-------+
|OrderID|ProductID|Quantity|CustomerID| OrderDate|PaymentMethod|UnitPrice|OrderStatus|TotalAmount|       CustomerEmail|     CustomerPhone|              City|         Country|       ProductName| Category|FirstName|Last Name|Year|Month|Day|Revenue|
+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+------------------+----------------+------------------+---------+---------+---------+----+-----+---+-------+
|  O0875|    P0001|       1|     C0421|2025-01-04|  Credit Card|   1491.0|  Cancelled|    4784.71|jessicavelazquez@...|      669.914.2368|   Port Danielview|         Bolivia|Interesting Tablet|Computers|    Brian|    White|2025|    1|  4| 1491.0|
|  O2741|   

In [18]:
df_new = df.filter(col('OrderStatus') != 'Cancelled')

In [19]:
df_new.show(5)

+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+----+-----+---+-------+
|OrderID|ProductID|Quantity|CustomerID| OrderDate|PaymentMethod|UnitPrice|OrderStatus|TotalAmount|       CustomerEmail|     CustomerPhone|             City|             Country|       ProductName| Category|FirstName|Last Name|Year|Month|Day|Revenue|
+-------+---------+--------+----------+----------+-------------+---------+-----------+-----------+--------------------+------------------+-----------------+--------------------+------------------+---------+---------+---------+----+-----+---+-------+
|  O0094|    P0001|       1|     C1954|2024-12-02|       PayPal|   1491.0|    Pending|    8476.45|griffinmichelle@e...|     (808)782-4405|       Emilymouth|Northern Mariana ...|Interesting Tablet|Computers|   Kelsey|   Burton|2024|   12|  2| 1491.0|


In [20]:
top_5 = (df.groupBy("CustomerID")
        .agg(sum("Revenue").alias ("TotalSpend"))
         .orderBy(col("TotalSpend").desc())
         .limit(5)                                       
)

top_5.show(5)

+----------+------------------+
|CustomerID|        TotalSpend|
+----------+------------------+
|     C0955|         517227.09|
|     C1995|464327.24000000005|
|     C1445|         464205.56|
|     C1802|         462075.91|
|     C1746|         461284.16|
+----------+------------------+



In [21]:
payment_usage_count = (df.groupBy("PaymentMethod")
        .agg(count("PaymentMethod").alias ("Count_of_PaymentMethod"))
        .orderBy(col("Count_of_PaymentMethod").desc()))

In [22]:
payment_usage_count.show()

+-------------+----------------------+
|PaymentMethod|Count_of_PaymentMethod|
+-------------+----------------------+
|   Debit Card|                 75309|
|  Credit Card|                 75145|
|Bank Transfer|                 75072|
|       PayPal|                 74268|
|      Unknown|                   172|
+-------------+----------------------+



In [23]:
total_payment_count = df.count()

In [24]:
payment_usage_count = (df.groupBy("PaymentMethod")
        .agg((round(count("PaymentMethod")/total_payment_count*100,2).alias ("percentage_of_PaymentMethod")))
        .orderBy(col("percentage_of_PaymentMethod").desc()))

In [25]:
payment_usage_count.show()

+-------------+---------------------------+
|PaymentMethod|percentage_of_PaymentMethod|
+-------------+---------------------------+
|   Debit Card|                      25.11|
|  Credit Card|                      25.05|
|Bank Transfer|                      25.03|
|       PayPal|                      24.76|
|      Unknown|                       0.06|
+-------------+---------------------------+

